In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader,Dataset

In [ ]:
data=pd.read_csv('/content/spam_ham_dataset.csv')

In [ ]:
data.head()

,Unnamed: 0,label,text,label_num
0,605,ham,Subject: enron methanol ; meter # : 988291\r\n...,0
1,2349,ham,"Subject: hpl nom for january 9 , 2001\r\n( see...",0
2,3624,ham,"Subject: neon retreat\r\nho ho ho , we ' re ar...",0
3,4685,spam,"Subject: photoshop , windows , office . cheap ...",1
4,2030,ham,Subject: re : indian springs\r\nthis deal is t...,0


In [ ]:
data.shape

(5171, 4)

In [ ]:
data['label'].value_counts()

,count
label,
ham,3672
spam,1499


In [ ]:
ham_msg=data[data['label']=='ham']
spam_msg=data[data['label']=='spam']
#downsample the ham emails to match spam emails
ham_msg_balanced=ham_msg.sample(n=len(spam_msg),random_state=42)

#combine balanced dataset
balanced_dataset=pd.concat([spam_msg,ham_msg_balanced]).reset_index(drop=True)

In [ ]:
balanced_dataset['label'].value_counts()

,count
label,
spam,1499
ham,1499


Data preprocessing

In [ ]:
import string

In [ ]:
def tokenize(text):

  #turns all data to lowercase
  text=text.lower()

  #removes the subject literal
  text=text.replace("subject:"," ")

  #removes the line break
  text=text.replace("\n"," ").replace("\t"," ")

  translator=text.maketrans(","," ",string.punctuation)
  text = text.translate(translator)

  # splits the data
  tokens=text.split()

  return tokens

In [ ]:
tokenize(balanced_dataset['text'][0])

['photoshop',
 'windows',
 'office',
 'cheap',
 'main',
 'trending',
 'abasements',
 'darer',
 'prudently',
 'fortuitous',
 'undergone',
 'lighthearted',
 'charm',
 'orinoco',
 'taster',
 'railroad',
 'affluent',
 'pornographic',
 'cuvier',
 'irvin',
 'parkhouse',
 'blameworthy',
 'chlorophyll',
 'robed',
 'diagrammatic',
 'fogarty',
 'clears',
 'bayda',
 'inconveniencing',
 'managing',
 'represented',
 'smartness',
 'hashish',
 'academies',
 'shareholders',
 'unload',
 'badness',
 'danielson',
 'pure',
 'caffein',
 'spaniard',
 'chargeable',
 'levin']

In [ ]:
balanced_dataset['tokens']=balanced_dataset['text'].apply(tokenize)

In [ ]:
balanced_dataset

,Unnamed: 0,label,text,label_num,tokens
0,4685,spam,"Subject: photoshop , windows , office . cheap ...",1,"[photoshop, windows, office, cheap, main, tren..."
1,4185,spam,Subject: looking for medication ? we ` re the ...,1,"[looking, for, medication, we, re, the, best, ..."
2,4922,spam,Subject: vocable % rnd - word asceticism\r\nvc...,1,"[vocable, rnd, word, asceticism, vcsc, brand, ..."
3,3799,spam,Subject: report 01405 !\r\nwffur attion brom e...,1,"[report, 01405, wffur, attion, brom, est, inst..."
4,3948,spam,Subject: vic . odin n ^ ow\r\nberne hotbox car...,1,"[vic, odin, n, ow, berne, hotbox, carnal, brid..."
...,...,...,...,...,...
2993,3370,ham,Subject: fw : duke energy trading and marketin...,0,"[fw, duke, energy, trading, and, marketing, l,..."
2994,180,ham,"Subject: re : greatwood gas\r\nthanks , kyle ....",0,"[re, greatwood, gas, thanks, kyle, greatwood, ..."
2995,686,ham,Subject: tufco\r\nmy est . for tufco is the sa...,0,"[tufco, my, est, for, tufco, is, the, same, as..."
2996,472,ham,Subject: mobil discrepancies\r\ndaren :\r\ni '...,0,"[mobil, discrepancies, daren, i, ll, come, dow..."


In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(balanced_dataset['tokens'],balanced_dataset['label_num'],test_size=0.2,random_state=42)

In [ ]:
from collections import Counter
def build_vocab(token_list,max_vocab_size=5000):
  counter=Counter()
  for token in token_list:
    counter.update(token)
  most_common=counter.most_common(max_vocab_size)
  vocab={'<unk>':1,
           '<pad>':0}
  for word,_ in most_common:
    if word not in vocab:
      vocab[word]=len(vocab)
  return vocab
vocab = build_vocab(X_train,max_vocab_size=5000)
print(f"Vocabulary size: {len(vocab)}")
print(f"Example tokens: {list(vocab.items())[:10]}")
print(f"Train tokens sample: {X_train.iloc[0]}")

Vocabulary size: 5002
Example tokens: [('<unk>', 1), ('<pad>', 0), ('the', 2), ('to', 3), ('and', 4), ('of', 5), ('a', 6), ('for', 7), ('ect', 8), ('you', 9)]
Train tokens sample: ['re', 'job', 'posting', 'daren', 'is', 'this', 'position', 'budgeted', 'and', 'who', 'does', 'it', 'report', 'to', 'thanks', 'toni', 'graham', 'x', '39995', 'daren', 'j', 'farmer', 'ect', '05', '30', '2000', '06', '30', 'am', 'to', 'toni', 'graham', 'corp', 'enron', 'enron', 'cc', 'subject', 'job', 'posting', 'toni', 'attached', 'is', 'the', 'job', 'posting', 'for', 'my', 'group', 'please', 'process', 'this', 'as', 'soon', 'as', 'possible', 'thanks', 'daren']


In [ ]:
def text_to_indices(tokens_series,vocab):
  all_indices=[]
  for token_list in tokens_series:
    email_indices=[]
    for token in token_list:
      if token in vocab:
        email_indices.append(vocab[token])
      else:
        email_indices.append(vocab['<unk>'])
    all_indices.append(email_indices)

  return all_indices

In [ ]:
len_vocab=text_to_indices(X_train,vocab)

In [ ]:
def pad_sequences(seq,max_len,pad_value):
  padded_seq=[]
  for s in seq:
    if len(s) < max_len:
      s=s+[pad_value]*(max_len-len(s))
    else:
      s=s[:max_len]

    padded_seq.append(s)
  return padded_seq

In [ ]:
MAX_LEN=300
train_padded=pad_sequences(text_to_indices(X_train, vocab),max_len=MAX_LEN,pad_value=vocab['<pad>'])
test_padded=pad_sequences(text_to_indices(X_test, vocab),max_len=MAX_LEN,pad_value=vocab['<pad>'])

In [ ]:
print("Length of first padded email:", len(train_padded[0]))
print("Sample padded email indices:", train_padded[0][:20])

Length of first padded email: 300
Sample padded email indices: [59, 2303, 1, 68, 11, 12, 543, 1, 4, 189, 336, 25, 168, 3, 70, 4089, 3179, 149, 1, 68]


In [ ]:
class EmailDataset(Dataset):
  def __init__(self,X,y):
    self.X=X
    self.y=y
  def __len__(self):
    return self.y.shape[0]

  def __getitem__(self,idx):
    x_tensor=torch.tensor(self.X[idx],dtype=torch.long)
    y_tensor=torch.tensor(self.y[idx],dtype=torch.float)
    return x_tensor,y_tensor

In [ ]:
# y_train_num=(y_train == 'spam').astype(int).tolist()
# y_test_num=(y_test == 'spam').astype(int).tolist()
y_train=y_train.tolist()
y_test=y_test.tolist()
train_dataset=EmailDataset(train_padded,y_train)
test_dataset=EmailDataset(test_padded,y_test)

In [ ]:
x_sample, y_sample = train_dataset[0]
print("Sample input shape:", x_sample.shape)
print("Sample label:", y_sample)

Sample input shape: torch.Size([300])
Sample label: tensor(0.)


In [ ]:
class SimpleRNN(nn.Module):
  def __init__(self,vocab_size,embed_dim,hidden_dim,output_dim,padding_idx):
    super().__init__()
    self.embedding=nn.Embedding(num_embeddings=vocab_size,embedding_dim=embed_dim,padding_idx=padding_idx)
    self.lstm=nn.LSTM(input_size=embed_dim,hidden_size=hidden_dim,batch_first=True)
    self.fc=nn.Linear(hidden_dim,output_dim)

  def forward(self,x):
    embedded=self.embedding(x)
    intermediate_hidden_state,(final_hidden_state,final_cell_state)=self.lstm(embedded)
    fc=self.fc(final_hidden_state)
    return fc

In [ ]:
model=SimpleRNN(vocab_size = len(vocab),
                embed_dim = 64,
                hidden_dim = 64,
                output_dim = 1,
                padding_idx = vocab["<pad>"])


In [ ]:
x=nn.Embedding(5002,64,1)
y=nn.LSTM(64,64,batch_first=True)
a=train_dataset[0][0].unsqueeze(0)
b=x(a)
print(b.shape)
c,d=y(b)
print(c.shape)
e,f=d
print(e.shape)
print(f.shape)
z=nn.Linear(64,1)
g=z(e).squeeze(0)
print(g.shape)

torch.Size([1, 300, 64])
torch.Size([1, 300, 64])
torch.Size([1, 1, 64])
torch.Size([1, 1, 64])
torch.Size([1, 1])


In [ ]:
learning_rate=0.001
epochs=20
loss_fn=nn.BCEWithLogitsLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=learning_rate)

In [ ]:
for epoch in range(epochs):
    total_loss = 0.0

    for x, y in train_dataset:
        x = x.unsqueeze(0)              # [1, seq_len]
        y = y.float().unsqueeze(0).unsqueeze(1)      # [1]

        pred = model(x)                 # [1, 1]
        pred = pred.squeeze(1)          # [1]

        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss:.4f}")


Epoch 1/20 | Loss: 1661.2740
Epoch 2/20 | Loss: 1426.3263
Epoch 3/20 | Loss: 1456.7485
Epoch 4/20 | Loss: 1301.3897
Epoch 5/20 | Loss: 1286.0927
Epoch 6/20 | Loss: 1205.3626
Epoch 7/20 | Loss: 1206.0427
Epoch 8/20 | Loss: 1147.1480
Epoch 9/20 | Loss: 1014.8887
Epoch 10/20 | Loss: 1188.4534
Epoch 11/20 | Loss: 1051.8382
Epoch 12/20 | Loss: 1083.3381
Epoch 13/20 | Loss: 1021.0424
Epoch 14/20 | Loss: 1054.3641
Epoch 15/20 | Loss: 653.7533
Epoch 16/20 | Loss: 348.8633
Epoch 17/20 | Loss: 267.3596
Epoch 18/20 | Loss: 201.9477
Epoch 19/20 | Loss: 162.9378
Epoch 20/20 | Loss: 123.7814


In [ ]:
model.eval()
with torch.inference_mode():
  correct=0
  total=0
  for x,y in test_dataset:
    x=x.unsqueeze(0)
    y=y.unsqueeze(0).unsqueeze(1)
    logits=model(x)
    prob=torch.sigmoid(logits)
    pred_label = (prob >= 0.5).long()
    correct +=(pred_label==y).sum().item()
    total+=y.size(0)
    accuracy=correct/total
  print(f"Test Accuracy: {accuracy*100:.2f}%")

Test Accuracy: 96.50%
